# 📊 Model vs SOMEF Evaluation Pipeline

This notebook evaluates the trained NER model against SOMEF framework on GitHub repositories.

## Pipeline Overview:
1. **GitHub README Extraction** - Retrieve README from GitHub repository
2. **Text Preprocessing** - Clean and split README using extraction pipeline utilities
3. **Model Inference** - Apply trained NER model to extract metadata
4. **SOMEF Extraction** - Use SOMEF framework to extract metadata from repository
5. **Ground Truth Comparison** - Compare against Label Studio annotations
6. **Metrics Calculation** - Precision, Recall, F1, and Term Coverage

## 1. Setup and Dependencies

In [ ]:
# Install dependencies
!pip install tqdm somef --quiet

import re
import json
import os
import logging
import tempfile
import subprocess
from typing import Dict, Any,List

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("SOMEF_EXTRACTION")


## 2. Configuration

In [ ]:
# Evaluation configuration
OUTPUT_DIR = "../evaluation/extrated_codemeta_files"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. SOMEF Metadata Extraction

In [ ]:
def run_somef_on_repository(repo_url: str) -> tuple[Dict[str, Any], Dict[str, Any]]:
    """Run SOMEF on GitHub repository and return both SOMEF and Codemeta results."""
    somef_data = {}
    codemeta_data = {}
    
    try:
        # Create temporary directory for SOMEF output
        with tempfile.TemporaryDirectory() as temp_dir:
            output_file = os.path.join(temp_dir, "somef_output.json")
            codemeta_file = os.path.join(temp_dir, "codemeta.json")
            
            # Run SOMEF command - separate output files for each format
            cmd = [
                "somef", "describe",
                "-r", repo_url,
                "-o", output_file,        # SOMEF JSON output
                "-c", codemeta_file,      # Codemeta output
                "-t", "0.8"  # Confidence threshold
            ]
            
            logger.info(f"Running SOMEF on {repo_url}...")
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
            
            if result.returncode != 0:
                logger.error(f"SOMEF failed: {result.stderr}")
                return {}, {}
            
            # Retrieve Codemeta output
            if os.path.exists(codemeta_file):
                with open(codemeta_file, 'r', encoding='utf-8') as f:
                    codemeta_data = json.load(f)
                logger.info("Codemeta extraction completed successfully")
            else:
                logger.error("Codemeta output file not found")
            
            # Read SOMEF output
            if os.path.exists(output_file):
                with open(output_file, 'r', encoding='utf-8') as f:
                    somef_data = json.load(f)
                logger.info("SOMEF extraction completed successfully")
            else:
                logger.error("SOMEF output file not found")
            
            return somef_data, codemeta_data
                
    except subprocess.TimeoutExpired:
        logger.error("SOMEF execution timed out")
        return {}, {}
    except Exception as e:
        logger.error(f"Error running SOMEF: {e}")
        return {}, {}
    

def filter_somef_readme_metadata(somef_data: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Extract all SOMEF entities where the source is the README.
    
    Returns a list of {field_name, value, confidence, source_url} for README-sourced entries.
    """
    readme_entities = []
    
    if not somef_data:
        return readme_entities
    
    # Iterate through all fields in SOMEF output
    for field_name, field_data in somef_data.items():
        # SOMEF returns lists for all field values
        if isinstance(field_data, list):
            for entry in field_data:
                if isinstance(entry, dict):
                    # Get source URL - it's a string, not a list
                    source = entry.get('source', '')
                    
                    # Only include if source is README
                    if isinstance(source, str) and 'readme' in source.lower():
                        # Extract the actual value/text
                        result_obj = entry.get('result', {})
                        text_value = None
                        
                        # Handle different result structures
                        if isinstance(result_obj, dict):
                            # Could be {'value': '...', 'type': '...'} or other formats
                            text_value = result_obj.get('value', '')
                        elif isinstance(result_obj, str):
                            text_value = result_obj
                        
                        # Fallback to excerpt if no result
                        if not text_value:
                            text_value = entry.get('excerpt', '')
                        
                        # Get confidence
                        confidence = entry.get('confidence', 1.0)
                        
                        if text_value:  # Only add if we have actual content
                            readme_entities.append({
                                'field': field_name,
                                'value': text_value,
                                'confidence': confidence,
                                'source': source,
                                'technique': entry.get('technique', ''),  # e.g., 'header_analysis', 'file_exploration'
                                'original_header': entry.get('result', {}).get('original_header', '') if isinstance(entry.get('result'), dict) else ''
                            })
    
    logger.info(f"Filtered {len(readme_entities)} README-sourced entities from SOMEF")
    return readme_entities

## 4. Filter Codemeta by README-sourced fields

In [ ]:
def filter_codemeta_by_readme(codemeta_data: Dict[str, Any], 
                                somef_entities: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Dynamically filter codemeta.json to only include fields whose values 
    match or contain values from README-sourced SOMEF entities.
    
    Args:
        codemeta_data: The full codemeta.json data
        somef_entities: List of README-sourced entities from SOMEF
    
    Returns:
        Filtered codemeta dictionary with only README-sourced fields
    """
    # Extract all values from README-sourced SOMEF entities
    readme_values = set()
    for entity in somef_entities:
        value = entity.get('value', '')
        if isinstance(value, str) and value.strip():
            readme_values.add(value.strip())
    
    logger.info(f"Extracted {len(readme_values)} unique values from README-sourced SOMEF entities")
    
    # Start with required codemeta fields
    filtered_codemeta = {
        "@context": codemeta_data.get("@context"),
        "@type": codemeta_data.get("@type")
    }
    
    def normalize_value(value):
        """Normalize a value for comparison."""
        if isinstance(value, str):
            return value.strip()
        return value
    
    def check_value_match(codemeta_value, readme_values) -> bool:
        """
        Recursively check if a codemeta value matches any README value.
        Handles strings, lists, dicts, and nested structures.
        """
        if isinstance(codemeta_value, str):
            normalized = normalize_value(codemeta_value)
            # Direct match
            if normalized in readme_values:
                return True
            # Check if any readme value is contained in this value
            for readme_val in readme_values:
                if readme_val in normalized or normalized in readme_val:
                    return True
            return False
        
        elif isinstance(codemeta_value, list):
            # If any item in the list matches, include the whole list
            return any(check_value_match(item, readme_values) for item in codemeta_value)
        
        elif isinstance(codemeta_value, dict):
            # Check all values in the dict
            return any(check_value_match(v, readme_values) for v in codemeta_value.values())
        
        return False
    
    # Filter codemeta fields dynamically
    for field_name, field_value in codemeta_data.items():
        # Skip already added required fields
        if field_name in ["@context", "@type"]:
            continue
        
        # Check if this field's value matches any README-sourced value
        if check_value_match(field_value, readme_values):
            filtered_codemeta[field_name] = field_value
            logger.info(f"Including field '{field_name}' - matched README value")
    
    return filtered_codemeta


## 5. Run batch extraction

In [ ]:
REPO_URLS = [
    "https://github.com/qc2nl/qc2",
    "https://github.com/resurfemg-org/ReSurfEMG",
    "https://github.com/eWaterCycle/Cesium-NcWMS",
    "https://github.com/MindTheGap-ERC/admtools",
    "https://github.com/sanctuuary/APE",
    "https://github.com/computationalgeography/lue",
    "https://github.com/opensim-org/opensim-core",
    "https://github.com/NNPDF/nnpdf",
    "https://github.com/SMEISEN/AutoPQ",
    "https://github.com/iBridges-for-iRODS/iBridges-GUI"
]

for REPO_URL in REPO_URLS:
    # Run SOMEF extraction
    somef_data, codemeta_data = run_somef_on_repository(REPO_URL)

    somef_entities = filter_somef_readme_metadata(somef_data)
    logger.info(f"Extracted {somef_data}")

    # Filter codemeta to only README-sourced fields
    filtered_codemeta = filter_codemeta_by_readme(codemeta_data, somef_entities)

    # Save extracted Codemeta data
    repo_name = REPO_URL.rstrip('/').split('/')[-1]
    output_path = os.path.join(OUTPUT_DIR, f"{repo_name}_somef_codemeta.json")
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(filtered_codemeta, f, indent=2)
    logger.info(f"Saved Codemeta data to {output_path}")

logger.info("SOMEF extraction and filtering completed.")